# Forecasting and dynamical assessment: Mackey–Glass and Kuramoto–Sivashinsky

This notebook implements the paper's predictor-comparison experiment and displays its measured results. It is separate from the archived forecasts and from both CAP certificates. **KS here is the length-22 periodic, 14-node reference; the Neumann CAP is a different boundary-value problem.**

The models are dense polynomial regression (DPSR), custom sparse STLSQ, a two-tap quadratic NVAR, a 100-unit echo-state network, and persistence. NVAR/ESN follow established model families, not exact replications of their published benchmark configurations.

Sources: [SINDy](https://doi.org/10.1073/pnas.1517384113), [next-generation reservoir computing](https://doi.org/10.1038/s41467-021-25801-2), [reservoir prediction of spatiotemporal chaos](https://doi.org/10.1103/PhysRevLett.120.024102).

## 1. Configuration and reproduction

The default displays the completed `paper` profile. Set `RUN=True` to regenerate it. `smoke` and `pilot` profiles use shorter rollouts and smaller grids; they must not be substituted silently into the paper table. The helper has a command-line entry point: `PYTHONPATH=benchmark/code python -m ddsr_benchmark.paper_comparison --profile paper --output results/predictor_comparison/paper`.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "benchmark/code/ddsr_benchmark").is_dir())
sys.path.insert(0, str(ROOT / "benchmark/code"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from ddsr_benchmark.paper_comparison import run
RUN = False
PROFILE = "paper"
OUTPUT = ROOT / "results/predictor_comparison" / PROFILE
if RUN:
    run(OUTPUT, PROFILE)
metadata = json.loads((OUTPUT / "metadata.json").read_text())
assert metadata["complete"], "Interrupted experiment: regenerate before using these results."
for path, digest in metadata.get("export_source_sha256", {}).items():
    assert hashlib.sha256((ROOT / path).read_bytes()).hexdigest() == digest, f"Source changed: {path}; rerun."
print("Profile:", metadata["profile"])
print("Scope:", metadata["scope"])


## 2. Information access and selection protocol

MG uses six delayed scalar coordinates (lag 4.5), a cubic vector field, and finite-difference derivative labels. Its new numerical reference uses Euler step 0.01 and observation interval 0.1. KS uses quadratic vector fields and exact derivative labels from the same 14-node spectral reference used for forecasts.

**The vector-field models have derivative labels; NVAR, ESN, and persistence use values only.** ESN receives 50 prior observations for initialization; NVAR uses the previous and current vectors; ODE models use the current vector. The same history is available to all methods, but is used differently. No future test observations enter a rollout.

Hyperparameters are selected on validation rollouts only, prioritizing numerical completion, failure-inclusive horizon, and then completed-rollout NRMSE. Reference attractor statistics are not used for selection. The ridge grid includes zero, which matters when recovering an already-quadratic spectral operator. ESN architecture is fixed and its seed is selected from the recorded grid. This is an exploratory comparison with unequal search costs, not a universal ranking.


In [ ]:
display(pd.DataFrame(metadata["datasets"])[["name", "boundary", "labels", "splits", "duration", "long_duration"]])
print("Ridge grid:", metadata["alphas"])
print("STLSQ thresholds:", metadata["thresholds"])
print("ESN seeds:", metadata["esn_seeds"])
print("Selection:", metadata["selection"])
summary = pd.read_csv(OUTPUT / "summary.csv")
sweep = pd.read_csv(OUTPUT / "validation_sweep.csv")
display(summary[["system", "family", "alpha", "threshold", "seed", "validation_completed", "validation_horizon"]])


## 3. Autonomous held-out forecasts

Horizon uses cumulative squared error (threshold 0.05 for MG, 0.5 for KS). NRMSE uses a fixed training standard deviation, so it differs from the archived notebook's per-rollout normalization. Escapes/nonfinite trajectories receive zero horizon and stay in the denominator. NRMSE is conditional on completed rollouts; missing entries are failures, not zero errors. Starts share one deterministic trajectory and can overlap, so they are not independent replicates.


In [ ]:
display(summary[["system", "family", "test_completed", "test_horizon", "test_nrmse", "fit_seconds", "rollout_seconds"]])
display(Image(filename=str(OUTPUT / "forecast_comparison.png")))
cases = pd.read_csv(OUTPUT / "forecast_cases.csv")
display(cases)


## 4. Individual forecast traces

The first test case is shown for each system. KS shows one node for readability; the reported KS error uses the entire spatial field. A line ending early represents a failed rollout and is not extended artificially.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for ax, slug, title in zip(axes, ["mackey_glass", "periodic_ks"], ["Mackey–Glass", "Periodic KS (node 0)"]):
    for family in ["DPSR", "SINDy-STLSQ", "NVAR", "ESN", "Persistence"]:
        z = np.load(OUTPUT / f"{slug}_{family.lower()}_rollouts.npz")
        t = np.arange(z["truth"].shape[1]) * float(z["dt"])
        if family == "DPSR":
            ax.plot(t, z["truth"][0, :, 0], color="black", lw=2, label="Reference")
        ax.plot(t, z["prediction"][0, :, 0], lw=1, alpha=.8, label=family)
    ax.set(title=title, xlabel="Time from initialization", ylabel="Observed value")
    ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()


## 5. Numerical attractor diagnostics

These are finite long-run diagnostics, not proofs of invariant-measure equality, boundedness for all initial conditions, or chaos. The first quarter of each rollout is discarded. MG statistics use the observed scalar; KS marginal statistics pool spatial nodes, and the spectrum averages nodewise power. Pooling can conceal spatial errors; a full spatial-correlation study is future work. Constant persistence can complete a run but does not reproduce the attractor.


In [ ]:
display(summary[["system", "family", "long_completed", "wasserstein", "std_ratio", "spectrum_l1"]])
display(pd.read_csv(OUTPUT / "long_run_cases.csv"))


## 6. Integration sensitivity and limitations

Selected ODE models are evaluated with RK4 internal steps 0.05 and 0.025. This checks one source of numerical sensitivity; it is not a convergence proof for the reference solver, the full PDE, or long-time statistics. Discrete NVAR/ESN models are defined at the observation step and do not share this integration test.

The grids and implementation were developed during pilot experiments. A new untouched reference trajectory is needed before making confirmatory performance claims. Additional work should sweep reservoir size/leak/spectral radius and NVAR tap spacing/degree, vary observation noise and derivative access, increase reference resolution, and report independent histories/seeds. Failed configurations remain in the saved validation sweep.


In [ ]:
display(pd.read_csv(OUTPUT / "integration_checks.csv"))
display(sweep)
print("Saved selected models, input datasets, all rollouts, and source hashes:", OUTPUT)


## 7. Relationship to CAP assessment

The forecasting model, orbit-discovery model, and certified candidate are separate objects. For MG, the archived cubic predictor and degree-five orbit-refinement model have different roles. For KS, these periodic-domain forecasts do not test the Neumann proof problem. A CAP radius measures proximity of a frozen candidate to an exact orbit in a specified norm; it is not the forecast error of any row above.

The saved certificates below are displayed for context and are **not rerun** by this notebook. This comparison does not certify periodic orbits for every baseline.


In [ ]:
mg_cap = json.loads((ROOT / "CAP/Mackey-Glass_PO/outputs_fixed_parameter/mg_fixed_parameter_certificate.json").read_text())
ks_cap = json.loads((ROOT / "CAP/Kuramoto_PO/outputs_surrogate/ks_neumann_full_pde_certificate.json").read_text())
print("MG saved certificate:")
display(mg_cap)
print("Neumann KS saved certificate (different boundary problem from forecasts):")
display(ks_cap)
